In [ ]:


# Step 1: Install required libraries
!pip install langgraph langchain-groq langchain-core -q

# Step 2: Import libraries
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from google.colab import userdata
import os
from IPython.display import Image, display
from getpass import getpass

print("✅ All libraries installed and imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
✅ All libraries installed and imported successfully!


In [ ]:


# Method 1: Get from Colab secrets (recommended)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ API key loaded from Colab secrets!")
except:
    # Method 2: Ask user for input
    GROQ_API_KEY = getpass("Enter your Groq API Key: ")
    print("✅ API key received!")

# Set environment variable
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize Groq LLM (using llama-3.3-70b which is very capable)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    groq_api_key=GROQ_API_KEY
)

print(f"✅ Groq LLM initialized: llama-3.3-70b-versatile")

Enter your Groq API Key: ··········
✅ API key received!
✅ Groq LLM initialized: llama-3.3-70b-versatile


In [ ]:
# ============================================
# 📚 PART 1: CORE CONCEPTS
# ============================================

print("="*60)
print("PART 1: Core Concepts")
print("="*60)

# What is LangGraph?
print("""
LangGraph extends LangChain to enable:
- Cycles and Branching: Unlike chains, graphs can have loops
- State Management: Built-in state persistence
- Human-in-the-Loop: Pause and resume workflows
- Streaming Support: Stream intermediate results

Key Components:
1. StateGraph: The main graph class
2. Nodes: Functions that process state
3. Edges: Connections between nodes
4. Conditional Edges: Dynamic routing based on state
5. State: Typed dictionary holding workflow data
""")

PART 1: Core Concepts

LangGraph extends LangChain to enable:
- Cycles and Branching: Unlike chains, graphs can have loops
- State Management: Built-in state persistence
- Human-in-the-Loop: Pause and resume workflows
- Streaming Support: Stream intermediate results

Key Components:
1. StateGraph: The main graph class
2. Nodes: Functions that process state
3. Edges: Connections between nodes
4. Conditional Edges: Dynamic routing based on state
5. State: Typed dictionary holding workflow data



In [ ]:
# ============================================
# 🎯 PART 2: YOUR FIRST GRAPH - SIMPLE CHATBOT
# ============================================

print("="*60)
print("PART 2: Simple Chatbot")
print("="*60)

# Step 1: Define the State
class State(TypedDict):
    messages: Annotated[list, add_messages]

print("✅ State schema defined!")

# Step 2: Define the Node Function
def chatbot(state: State):
    """Simple chatbot node that calls Groq's LLM."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

print("✅ Chatbot node defined!")

# Step 3: Build the Graph
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# Compile the graph
simple_graph = graph_builder.compile()

print("✅ Graph compiled successfully!")

# Step 4: Run the Graph
def chat(user_input: str):
    """Helper function to chat with our graph."""
    result = simple_graph.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    return result["messages"][-1].content

# Test it out
print("\n🔄 Testing the chatbot...")
response = chat("Hello! What is LangGraph? Explain briefly.")
print(f"\n🤖 Groq AI: {response[:300]}...")

PART 2: Simple Chatbot
✅ State schema defined!
✅ Chatbot node defined!
✅ Graph compiled successfully!

🔄 Testing the chatbot...

🤖 Groq AI: LangGraph is an open-source, large language model developed by a team of researchers. It's designed to process and generate human-like language, similar to other models like LLaMA or PaLM. LangGraph is trained on a massive dataset of text from the internet and can be fine-tuned for various natural l...


In [ ]:
# ============================================
# 🔄 PART 3: CONDITIONAL ROUTING
# ============================================

print("="*60)
print("PART 3: Conditional Routing")
print("="*60)

# Define state for routing example
class RoutingState(TypedDict):
    messages: Annotated[list, add_messages]
    intent: str

# Node 1: Classify intent
def classify_intent(state: RoutingState):
    """Classify user's intent."""
    classification_prompt = f"""Classify this message into ONE category:
- 'math' if it's a math question
- 'weather' if it's about weather
- 'general' for everything else

User message: {state['messages'][-1].content}

Respond with ONLY the category name."""

    response = llm.invoke([HumanMessage(content=classification_prompt)])
    intent = response.content.strip().lower()
    print(f"   📍 Detected intent: {intent}")
    return {"intent": intent}

# Node 2: Handle math questions
def handle_math(state: RoutingState):
    """Process math questions."""
    math_prompt = f"""You are a math tutor. Solve this step by step:
{state['messages'][-1].content}"""

    response = llm.invoke([HumanMessage(content=math_prompt)])
    return {"messages": [response]}

# Node 3: Handle weather questions
def handle_weather(state: RoutingState):
    """Process weather questions."""
    response = AIMessage(content="I don't have real-time weather data, but I can help you find weather information! Check weather.com for accurate forecasts.")
    return {"messages": [response]}

# Node 4: Handle general questions
def handle_general(state: RoutingState):
    """Process general questions."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

print("✅ Routing nodes defined!")

# Define the routing function
def route_based_on_intent(state: RoutingState) -> str:
    """Route to the appropriate handler based on intent."""
    intent = state.get("intent", "general")
    if "math" in intent:
        return "handle_math"
    elif "weather" in intent:
        return "handle_weather"
    else:
        return "handle_general"

# Build the routing graph
routing_builder = StateGraph(RoutingState)

# Add nodes
routing_builder.add_node("classify_intent", classify_intent)
routing_builder.add_node("handle_math", handle_math)
routing_builder.add_node("handle_weather", handle_weather)
routing_builder.add_node("handle_general", handle_general)

# Add edges
routing_builder.add_edge(START, "classify_intent")

# Add conditional edge
routing_builder.add_conditional_edges(
    "classify_intent",
    route_based_on_intent,
    {
        "handle_math": "handle_math",
        "handle_weather": "handle_weather",
        "handle_general": "handle_general"
    }
)

# All handlers go to END
routing_builder.add_edge("handle_math", END)
routing_builder.add_edge("handle_weather", END)
routing_builder.add_edge("handle_general", END)

# Compile
routing_graph = routing_builder.compile()

print("✅ Routing graph compiled!")

# Test the routing graph
test_messages = [
    "What is 25 * 17?",
    "What's the weather like?",
    "Tell me about LangGraph"
]

for msg in test_messages:
    print(f"\n{'='*50}")
    print(f"👤 User: {msg}")
    result = routing_graph.invoke({"messages": [HumanMessage(content=msg)]})
    print(f"🤖 AI: {result['messages'][-1].content[:150]}...")

PART 3: Conditional Routing
✅ Routing nodes defined!
✅ Routing graph compiled!

👤 User: What is 25 * 17?
   📍 Detected intent: math
🤖 AI: To solve this problem, we'll follow the order of operations and multiply 25 by 17.

Step 1: Multiply the numbers
25 * 17 = ?

To make it easier, we ca...

👤 User: What's the weather like?
   📍 Detected intent: weather
🤖 AI: I don't have real-time weather data, but I can help you find weather information! Check weather.com for accurate forecasts....

👤 User: Tell me about LangGraph
   📍 Detected intent: general
🤖 AI: LangGraph is an open-source, multilingual, large language model developed by Meta AI. It's designed to process and understand human language, generati...


In [ ]:
# ============================================
# 💾 PART 4: PERSISTENCE & MEMORY
# ============================================

print("="*60)
print("PART 4: Persistence & Memory")
print("="*60)

# Create a memory saver for checkpointing
memory = MemorySaver()

# Rebuild chatbot with memory
persistent_builder = StateGraph(State)
persistent_builder.add_node("chatbot", chatbot)
persistent_builder.add_edge(START, "chatbot")
persistent_builder.add_edge("chatbot", END)

# Compile with checkpointer
persistent_graph = persistent_builder.compile(checkpointer=memory)

print("✅ Persistent graph compiled!")

# Test persistence with a conversation
config = {"configurable": {"thread_id": "conversation-1"}}

# Message 1
result1 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Sarah.")]},
    config=config
)
print(f"\nTurn 1 - AI: {result1['messages'][-1].content}")

# Message 2 - The graph remembers!
result2 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config
)
print(f"Turn 2 - AI: {result2['messages'][-1].content}")

# New thread - No memory
new_config = {"configurable": {"thread_id": "conversation-2"}}
result3 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=new_config
)
print(f"Turn 3 (new thread) - AI: {result3['messages'][-1].content}")

PART 4: Persistence & Memory
✅ Persistent graph compiled!

Turn 1 - AI: Hello Sarah. It's nice to meet you. Is there something I can help you with or would you like to chat?
Turn 2 - AI: Your name is Sarah.
Turn 3 (new thread) - AI: I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation.


In [ ]:
# ============================================
# 🎮 PART 5: HUMAN-IN-THE-LOOP
# ============================================

print("="*60)
print("PART 5: Human-in-the-Loop")
print("="*60)

class ApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    action: str
    approved: bool

def propose_action(state: ApprovalState):
    """AI proposes an action."""
    return {
        "action": "Send email to team about project update",
        "messages": [AIMessage(content="📧 I'd like to send an email update about the project. Should I proceed?")]
    }

def execute_action(state: ApprovalState):
    """Execute the approved action."""
    if state.get("approved", False):
        return {"messages": [AIMessage(content=f"✅ Action executed: {state['action']}")]
    }
    else:
        return {"messages": [AIMessage(content="❌ Action cancelled by user.")]}

# Build graph with interrupt
hitl_builder = StateGraph(ApprovalState)
hitl_builder.add_node("propose_action", propose_action)
hitl_builder.add_node("execute_action", execute_action)

hitl_builder.add_edge(START, "propose_action")
hitl_builder.add_edge("propose_action", "execute_action")
hitl_builder.add_edge("execute_action", END)

# Compile with interrupt before execute_action
hitl_graph = hitl_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["execute_action"]
)

print("✅ Human-in-the-loop graph compiled!")

# Run until interrupt
config = {"configurable": {"thread_id": "hitl-demo"}}

print("\n🔄 Running graph...")
result = hitl_graph.invoke({"messages": []}, config=config)
print(f"\n🤖 AI: {result['messages'][-1].content}")
print(f"📝 Proposed action: {result['action']}")
print("\n⏸️ Execution paused. Waiting for human approval...")

# Simulate user approval
print("\n👤 Human: Approving action...")
result = hitl_graph.invoke({"approved": True}, config=config)
print(f"🤖 AI: {result['messages'][-1].content}")

PART 5: Human-in-the-Loop
✅ Human-in-the-loop graph compiled!

🔄 Running graph...

🤖 AI: 📧 I'd like to send an email update about the project. Should I proceed?
📝 Proposed action: Send email to team about project update

⏸️ Execution paused. Waiting for human approval...

👤 Human: Approving action...
🤖 AI: 📧 I'd like to send an email update about the project. Should I proceed?


In [ ]:
# ============================================
# 🤖 PART 6: BUILDING AN AGENT WITH TOOLS
# ============================================

print("="*60)
print("PART 6: Agent with Tools")
print("="*60)

# Define tools
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@tool
def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    if b == 0:
        return "Error: Division by zero"
    return a / b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

tools = [multiply, add, divide, subtract]
print(f"✅ Defined {len(tools)} tools: multiply, add, divide, subtract")

# Create LLM with tools bound
llm_with_tools = llm.bind_tools(tools)

# Define the agent node
def agent(state: State):
    """Call the LLM to decide what to do."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Routing function
def should_continue(state: State) -> str:
    """Determine if we should use tools or end."""
    last_message = state["messages"][-1]

    # If the LLM makes a tool call, route to tools
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    # Otherwise, end
    return "end"

print("✅ Agent logic defined!")

# Build the agent graph
agent_builder = StateGraph(State)

# Add nodes
agent_builder.add_node("agent", agent)
agent_builder.add_node("tools", ToolNode(tools))

# Add edges
agent_builder.add_edge(START, "agent")

# Conditional edge from agent
agent_builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

# After tools, go back to agent
agent_builder.add_edge("tools", "agent")

# Compile
agent_graph = agent_builder.compile()

print("✅ Agent graph compiled!")

# Test the agent
def run_agent(query: str):
    """Run the agent with a query."""
    print(f"\n{'='*50}")
    print(f"👤 User: {query}")
    print("-"*50)

    result = agent_graph.invoke({
        "messages": [HumanMessage(content=query)]
    })

    # Print the conversation
    for msg in result["messages"]:
        if isinstance(msg, HumanMessage):
            pass  # Already printed
        elif isinstance(msg, AIMessage):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                print(f"🤖 Agent: Using tools...")
                for tool_call in msg.tool_calls:
                    print(f"   🔧 {tool_call['name']}({tool_call['args']})")
            else:
                print(f"🤖 Agent: {msg.content}")
        else:
            # Tool message
            if hasattr(msg, 'content'):
                print(f"   ✓ Result: {msg.content}")

# Run some tests
run_agent("What is 25 multiplied by 17, then add 100?")
run_agent("Calculate (50 / 2) + (10 * 3) - 5")

PART 6: Agent with Tools
✅ Defined 4 tools: multiply, add, divide, subtract
✅ Agent logic defined!
✅ Agent graph compiled!

👤 User: What is 25 multiplied by 17, then add 100?
--------------------------------------------------
🤖 Agent: Using tools...
   🔧 multiply({'a': 25, 'b': 17})
   🔧 add({'a': 425, 'b': 100})
   ✓ Result: 425.0
   ✓ Result: 525.0
🤖 Agent: The result of 25 multiplied by 17 is 425, then adding 100 is 525.

👤 User: Calculate (50 / 2) + (10 * 3) - 5
--------------------------------------------------


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=multiply [{"a": 10, "b": 3}]</function>\n'}}

In [ ]:
# ============================================
# 🌐 PART 7: STREAMING
# ============================================

print("="*60)
print("PART 7: Streaming")
print("="*60)

print("\n🔄 Streaming agent execution:\n")

for event in agent_graph.stream(
    {"messages": [HumanMessage(content="Calculate 123 * 456")]},
    stream_mode="values"
):
    last_message = event["messages"][-1]

    if isinstance(last_message, AIMessage):
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"🔧 Calling tool: {last_message.tool_calls[0]['name']}")
        else:
            print(f"✅ Final answer: {last_message.content}")
    else:
        print(f"📊 Tool result: {last_message.content}")

PART 7: Streaming

🔄 Streaming agent execution:

📊 Tool result: Calculate 123 * 456
🔧 Calling tool: multiply
📊 Tool result: 56088.0
✅ Final answer: The result of 123 * 456 is 56088.


In [ ]:
# ============================================
# 🔁 PART 8: CYCLES AND LOOPS (RESEARCH LOOP)
# ============================================

print("="*60)
print("PART 8: Cycles and Loops - Research Agent")
print("="*60)

class ResearchState(TypedDict):
    question: str
    iterations: int
    max_iterations: int
    findings: list
    final_answer: str

def research_step(state: ResearchState):
    """Perform one research step."""
    prompt = f"""Question: {state['question']}
Previous findings: {state.get('findings', [])}

Provide ONE new insight or finding about this topic. Be brief (1-2 sentences)."""

    response = llm.invoke([HumanMessage(content=prompt)])

    new_findings = state.get('findings', []) + [response.content]

    print(f"   🔍 Finding {len(new_findings)}: {response.content[:100]}...")

    return {
        "findings": new_findings,
        "iterations": state.get('iterations', 0) + 1
    }

def synthesize(state: ResearchState):
    """Synthesize all findings into final answer."""
    findings_text = "\n".join([f"{i+1}. {f}" for i, f in enumerate(state['findings'])])

    prompt = f"""Question: {state['question']}

Research findings:
{findings_text}

Provide a comprehensive answer (2-3 paragraphs) based on these findings."""

    response = llm.invoke([HumanMessage(content=prompt)])

    return {"final_answer": response.content}

def should_continue_research(state: ResearchState) -> str:
    """Decide if we should continue researching."""
    if state.get('iterations', 0) >= state.get('max_iterations', 3):
        return "synthesize"
    return "research_step"

# Build the research graph with a cycle
research_builder = StateGraph(ResearchState)

research_builder.add_node("research_step", research_step)
research_builder.add_node("synthesize", synthesize)

research_builder.add_edge(START, "research_step")

# Conditional edge creates the loop
research_builder.add_conditional_edges(
    "research_step",
    should_continue_research,
    {
        "research_step": "research_step",  # Loop back!
        "synthesize": "synthesize"
    }
)

research_builder.add_edge("synthesize", END)

research_graph = research_builder.compile()

print("✅ Research graph with cycles compiled!\n")

# Test the research loop
print("🔄 Running research agent...\n")
result = research_graph.invoke({
    "question": "What are the key benefits of using LangGraph for AI applications?",
    "max_iterations": 3,
    "iterations": 0,
    "findings": []
})

print(f"\n{'='*60}")
print("📝 FINAL RESEARCH REPORT")
print("="*60)
print(f"\nQuestion: {result['question']}")
print(f"\nIterations completed: {result['iterations']}")
print(f"\nKey Findings:")
for i, finding in enumerate(result['findings'], 1):
    print(f"\n{i}. {finding}")
print(f"\n\nFinal Answer:\n{result['final_answer']}")

PART 8: Cycles and Loops - Research Agent
✅ Research graph with cycles compiled!

🔄 Running research agent...

   🔍 Finding 1: One key benefit of using LangGraph for AI applications is its ability to efficiently process and rep...
   🔍 Finding 2: Another key benefit of using LangGraph for AI applications is its ability to capture contextual depe...
   🔍 Finding 3: A new key benefit of using LangGraph for AI applications is its potential to facilitate transfer lea...

📝 FINAL RESEARCH REPORT

Question: What are the key benefits of using LangGraph for AI applications?

Iterations completed: 3

Key Findings:

1. One key benefit of using LangGraph for AI applications is its ability to efficiently process and represent complex linguistic relationships, enabling more accurate and informative natural language processing tasks. This is particularly useful for applications such as text summarization, sentiment analysis, and language translation, where nuanced understanding of language is crucia

In [ ]:
# ============================================
# 🎓 PART 9: MAP-REDUCE PATTERN
# ============================================

print("="*60)
print("PART 9: Map-Reduce Pattern")
print("="*60)

class MapReduceState(TypedDict):
    documents: list
    summaries: list
    final_summary: str

def summarize_document(doc: str) -> str:
    """Summarize a single document."""
    response = llm.invoke([HumanMessage(content=f"Summarize this in ONE sentence: {doc}")])
    return response.content

def map_step(state: MapReduceState):
    """Map: Summarize each document."""
    print("   📝 Summarizing documents...")
    summaries = [summarize_document(doc) for doc in state['documents']]
    for i, summary in enumerate(summaries, 1):
        print(f"      Doc {i}: {summary[:80]}...")
    return {"summaries": summaries}

def reduce_step(state: MapReduceState):
    """Reduce: Combine all summaries."""
    combined = "\n".join([f"- {s}" for s in state['summaries']])
    prompt = f"""Combine these summaries into ONE coherent paragraph summary:

{combined}"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_summary": response.content}

# Build map-reduce graph
mr_builder = StateGraph(MapReduceState)
mr_builder.add_node("map", map_step)
mr_builder.add_node("reduce", reduce_step)
mr_builder.add_edge(START, "map")
mr_builder.add_edge("map", "reduce")
mr_builder.add_edge("reduce", END)

map_reduce_graph = mr_builder.compile()

print("✅ Map-reduce graph compiled!\n")

# Test map-reduce
docs = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain's capabilities.",
    "Key features of LangGraph include support for cycles, conditional routing, persistence with checkpointing, and streaming.",
    "LangGraph enables complex patterns like human-in-the-loop workflows, multi-agent systems, and map-reduce operations."
]

print("🔄 Running map-reduce...\n")
result = map_reduce_graph.invoke({"documents": docs})

print(f"\n{'='*50}")
print("📊 MAP-REDUCE RESULTS")
print("="*50)
print(f"\nIndividual Summaries:")
for i, summary in enumerate(result['summaries'], 1):
    print(f"{i}. {summary}")

print(f"\n\nFinal Combined Summary:\n{result['final_summary']}")

PART 9: Map-Reduce Pattern
✅ Map-reduce graph compiled!

🔄 Running map-reduce...

   📝 Summarizing documents...
      Doc 1: LangGraph is a library that builds upon LangChain to enable the development of s...
      Doc 2: LangGraph is a system that offers several key features, including support for cy...
      Doc 3: LangGraph enables complex patterns such as human-in-the-loop workflows, multi-ag...

📊 MAP-REDUCE RESULTS

Individual Summaries:
1. LangGraph is a library that builds upon LangChain to enable the development of stateful, multi-actor applications using large language models (LLMs).
2. LangGraph is a system that offers several key features, including support for cycles, conditional routing, persistence with checkpointing, and streaming.
3. LangGraph enables complex patterns such as human-in-the-loop workflows, multi-agent systems, and map-reduce operations, allowing for advanced and dynamic processing capabilities.


Final Combined Summary:
LangGraph is a library built upon 

In [ ]:
# ============================================
# 🎓 PART 10: MULTI-AGENT COLLABORATION
# ============================================

print("="*60)
print("PART 10: Multi-Agent Collaboration")
print("="*60)

class MultiAgentState(TypedDict):
    topic: str
    research_findings: list
    writer_draft: str
    critic_feedback: str
    final_article: str
    current_agent: str

def researcher_agent(state: MultiAgentState):
    """Agent 1: Researches the topic."""
    prompt = f"""Topic: {state['topic']}

As a RESEARCHER, provide 3 key facts or insights about this topic. Be factual and specific."""

    response = llm.invoke([HumanMessage(content=prompt)])
    findings = response.content.split('\n')
    print(f"   📚 Researcher found {len([f for f in findings if f.strip()])} key insights")
    return {"research_findings": findings, "current_agent": "writer"}

def writer_agent(state: MultiAgentState):
    """Agent 2: Writes based on research."""
    research_text = "\n".join(state.get('research_findings', []))

    prompt = f"""Topic: {state['topic']}
Research findings:
{research_text}

As a WRITER, write a short paragraph (3-4 sentences) synthesizing this into engaging content."""

    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"   ✍️ Writer created a draft")
    return {"writer_draft": response.content, "current_agent": "critic"}

def critic_agent(state: MultiAgentState):
    """Agent 3: Critiques and improves."""
    prompt = f"""Original draft:
{state['writer_draft']}

As a CRITIC, provide:
1. One thing that works well
2. One suggestion for improvement
3. The improved version"""

    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"   🎯 Critic provided feedback")
    return {"critic_feedback": response.content, "current_agent": "finalizer"}

def finalizer_agent(state: MultiAgentState):
    """Agent 4: Finalizes the article."""
    prompt = f"""Using the critic's feedback, produce the FINAL version:

Feedback:
{state['critic_feedback']}

Return only the final article, ready for publication."""

    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"   ✨ Finalizer completed the article\n")
    return {"final_article": response.content}

# Build multi-agent graph
multi_builder = StateGraph(MultiAgentState)

# Add agents
multi_builder.add_node("researcher", researcher_agent)
multi_builder.add_node("writer", writer_agent)
multi_builder.add_node("critic", critic_agent)
multi_builder.add_node("finalizer", finalizer_agent)

# Define flow
multi_builder.add_edge(START, "researcher")
multi_builder.add_edge("researcher", "writer")
multi_builder.add_edge("writer", "critic")
multi_builder.add_edge("critic", "finalizer")
multi_builder.add_edge("finalizer", END)

multi_agent_graph = multi_builder.compile()

print("✅ Multi-agent graph compiled!\n")

# Test multi-agent system
print("🔄 Running multi-agent collaboration...\n")
print("="*50)

result = multi_agent_graph.invoke({
    "topic": "The benefits of learning AI in 2025"
})

print(f"\n{'='*50}")
print("📝 FINAL ARTICLE")
print("="*50)
print(result['final_article'])
print("\n" + "="*50)
print("✅ Multi-agent collaboration complete!")

PART 10: Multi-Agent Collaboration
✅ Multi-agent graph compiled!

🔄 Running multi-agent collaboration...

   📚 Researcher found 4 key insights
   ✍️ Writer created a draft
   🎯 Critic provided feedback
   ✨ Finalizer completed the article


📝 FINAL ARTICLE
As we step into 2025, learning AI is no longer a luxury, but a necessity for professionals and individuals looking to stay ahead of the curve. With the World Economic Forum predicting over 97 million new job opportunities in AI and machine learning, acquiring AI skills can significantly boost employability and career prospects. For instance, companies like Amazon and Walmart are already using AI-powered chatbots to automate customer service, freeing up human representatives to focus on more complex issues and improving overall customer satisfaction. By embracing AI, individuals can unlock new opportunities, drive business growth, and stay competitive in a rapidly evolving job market. Moreover, AI can streamline processes and drive in

In [ ]:
# ============================================
# 📚 BEST PRACTICES & QUICK REFERENCE
# ============================================

print("="*60)
print("BEST PRACTICES & QUICK REFERENCE")
print("="*60)

print("""
Key Takeaways:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. STATE DESIGN
   • Keep state minimal and well-typed
   • Use TypedDict with Annotated types for reducers
   • Example: messages: Annotated[list, add_messages]

2. NODE FUNCTIONS
   • Pure functions that return state updates
   • Don't modify input state directly
   • Return dict with only changed fields

3. COMMON PATTERNS
   • Linear Pipeline: START → Node1 → Node2 → END
   • Router: START → Classifier → [Handlers] → END
   • Loop: START → Process → [Continue→Process | End→END]
   • Human-in-the-Loop: START → Propose → [Approval] → Execute → END
   • Agent: START → Agent → [Tools→Agent | END]

4. PERFORMANCE TIPS
   • Batch operations when possible
   • Use streaming for long-running operations
   • Implement retry logic in nodes
   • Cache expensive computations

5. DEBUGGING
   # Enable debug logging
   import logging
   logging.basicConfig(level=logging.DEBUG)

   # Stream state
   for step in graph.stream(initial_state):
       print(f"State: {step}")
""")

print("\n" + "="*60)
print("ESSENTIAL IMPORTS FOR REFERENCE")
print("="*60)

print("""
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

# Initialize Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    groq_api_key="your-api-key"
)
""")

print("\n" + "="*60)
print("BASIC GRAPH TEMPLATE")
print("="*60)

print("""
# 1. Define State
class State(TypedDict):
    field: type

# 2. Define Nodes
def node_function(state: State):
    # Process state
    return {"field": new_value}

# 3. Build Graph
builder = StateGraph(State)
builder.add_node("node_name", node_function)
builder.add_edge(START, "node_name")
builder.add_edge("node_name", END)

# 4. Compile & Run
graph = builder.compile()
result = graph.invoke(initial_state)
""")

BEST PRACTICES & QUICK REFERENCE

Key Takeaways:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. STATE DESIGN
   • Keep state minimal and well-typed
   • Use TypedDict with Annotated types for reducers
   • Example: messages: Annotated[list, add_messages]

2. NODE FUNCTIONS
   • Pure functions that return state updates
   • Don't modify input state directly
   • Return dict with only changed fields

3. COMMON PATTERNS
   • Linear Pipeline: START → Node1 → Node2 → END
   • Router: START → Classifier → [Handlers] → END
   • Loop: START → Process → [Continue→Process | End→END]
   • Human-in-the-Loop: START → Propose → [Approval] → Execute → END
   • Agent: START → Agent → [Tools→Agent | END]

4. PERFORMANCE TIPS
   • Batch operations when possible
   • Use streaming for long-running operations
   • Implement retry logic in nodes
   • Cache expensive computations

5. DEBUGGING
   # Enable debug logging
   import logging
   logging.basicConfig(level=logging.DEBUG)
   

In [ ]:


print("="*60)
print("BONUS: Interactive Essay Writer Agent")
print("="*60)

class EssayState(TypedDict):
    topic: str
    outline: str
    draft: str
    final_essay: str

def create_outline(state: EssayState):
    """Create essay outline."""
    prompt = f"""Create a brief outline for an essay on: {state['topic']}

Format:
I. Introduction with thesis
II. Main point 1
III. Main point 2
IV. Main point 3
V. Conclusion"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"outline": response.content}

def write_draft(state: EssayState):
    """Write essay draft."""
    prompt = f"""Write a short essay (3-4 paragraphs) following this outline:

OUTLINE:
{state['outline']}

Write an engaging, well-structured essay."""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"draft": response.content}

def polish_essay(state: EssayState):
    """Polish final essay."""
    prompt = f"""Polish this essay for clarity and style:

{state['draft']}

Improve flow and fix any issues. Return the polished version only."""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"final_essay": response.content}

# Build essay writer graph
essay_builder = StateGraph(EssayState)
essay_builder.add_node("outline", create_outline)
essay_builder.add_node("draft", write_draft)
essay_builder.add_node("polish", polish_essay)

essay_builder.add_edge(START, "outline")
essay_builder.add_edge("outline", "draft")
essay_builder.add_edge("draft", "polish")
essay_builder.add_edge("polish", END)

essay_writer = essay_builder.compile()

print("✅ Essay writer agent compiled!\n")

# Run interactive essay writer
topic = input("Enter an essay topic: ")

print(f"\n🚀 Writing essay about: {topic}\n")
print("="*50)

result = essay_writer.invoke({"topic": topic})

print(f"\n📋 OUTLINE:\n{result['outline']}\n")
print(f"✍️ DRAFT:\n{result['draft']}\n")
print(f"✨ FINAL ESSAY:\n{result['final_essay']}\n")
print("="*50)
print("✅ Essay complete!")

BONUS: Interactive Essay Writer Agent
✅ Essay writer agent compiled!

Enter an essay topic: egypt

🚀 Writing essay about: egypt


📋 OUTLINE:
Here is a brief outline for an essay on Egypt:

I. Introduction with thesis
    * Introduce the country of Egypt and its rich history
    * Thesis statement: Egypt is a unique and fascinating country with a blend of ancient traditions, stunning natural beauty, and modern culture, making it a captivating destination for tourists and a significant player in global affairs.

II. Main point 1: Ancient History and Architecture
    * Discuss the ancient pyramids, temples, and tombs, including the Great Pyramid of Giza and the Valley of the Kings
    * Describe the significance of these structures and their impact on Egyptian culture and identity

III. Main point 2: Natural Beauty and Geography
    * Describe the Nile River and its importance to Egyptian life and economy
    * Discuss the country's diverse landscapes, including deserts, mountains, and co

In [ ]:

# ============================================
# 🌐 BONUS: GRADIO WEB UI
# ============================================

!pip install -q gradio

import gradio as gr

print("🚀 Starting Gradio Interface...")

# Simple chatbot wrapper
def chat_with_agent(message):
    try:
        response = model.invoke(message)
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio Interface
demo = gr.Interface(
    fn=chat_with_agent,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type your message here...",
        label="Your Message"
    ),
    outputs=gr.Textbox(
        lines=8,
        label="AI Response"
    ),
    title="LangGraph + Groq Chatbot",
    description="Interactive AI chatbot using LangGraph, Groq, and Gradio",
    theme="soft"
)

demo.launch(share=True)


🚀 Starting Gradio Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4833474c41cd56b34.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
